# Exercícios 01–06 · Heterocedasticidade e erros-padrão robustos

**Prof. Daniel de Abreu Pereira Uhr**

**Curso:** Estatística e Econometria com Python


## Contexto — Indústria Pica-pau

No Capítulo 6, o laboratório da Pica-pau gerou 20.000 lotes de móveis com custo = R\$ 6.000 + R\$ 12 por unidade + um erro com desvio-padrão constante de R\$ 2.000. Agora, o laboratório mantém os mesmos tamanhos de lote e os mesmos choques, mas muda a **dispersão** do erro em três versões:

- **constante:** desvio-padrão de R\$ 2.000 — o laboratório do Capítulo 6;
- **crescente:** desvio-padrão de $0{,}0015 \cdot \text{unidades}^2$, de R\$ 60 nos lotes de 200 unidades a R\$ 6.000 nos de 2.000 — lotes grandes envolvem mais etapas e imprevistos;
- **maior no meio:** desvio-padrão de $3.500 - 3 \cdot |\text{unidades} - 1.100|$, máximo nos lotes médios (usado no desafio).

O analista observa 50 lotes — os mesmos do Capítulo 6 — com os custos da versão **crescente**.

Execute a célula abaixo antes de começar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

rng = np.random.default_rng(2026)
N = 20_000
unidades = rng.integers(200, 2_001, size=N)
choque = rng.normal(0, 1, size=N)
custo_constante = np.round(6_000 + 12 * unidades + 2_000 * choque, 2)
custo_crescente = np.round(6_000 + 12 * unidades + 0.0015 * unidades**2 * choque, 2)
custo_meio = np.round(6_000 + 12 * unidades + (3_500 - 3 * np.abs(unidades - 1_100)) * choque, 2)

n = 50
amostra = rng.choice(N, size=n, replace=True)
lotes = pd.DataFrame({"unidades": unidades[amostra], "custo": custo_crescente[amostra]})


def erros_padrao_simples(x, y):
    desvio_x = x - x.mean()
    soma_quadrados_x = (desvio_x**2).sum()
    inclinacao = (desvio_x * (y - y.mean())).sum() / soma_quadrados_x
    residuos = y - y.mean() - inclinacao * desvio_x
    n_obs = len(x)
    usual = np.sqrt((residuos**2).sum() / (n_obs - 2) / soma_quadrados_x)
    hc0 = np.sqrt((desvio_x**2 * residuos**2).sum() / soma_quadrados_x**2)
    hc1 = hc0 * np.sqrt(n_obs / (n_obs - 2))
    return inclinacao, usual, hc0, hc1

## Exercício 01 — Diagnóstico

Estime a regressão do custo nas unidades com os 50 lotes do analista. Faça o gráfico dos resíduos contra os valores ajustados e calcule o desvio-padrão dos resíduos nas faixas de 200 a 650, 650 a 1.100, 1.100 a 1.550 e 1.550 a 2.000 unidades. O que você conclui?

In [ ]:
# Escreva sua solução aqui

## Exercício 02 — Erros-padrão à mão

Use a função `erros_padrao_simples` para calcular os erros-padrão usual, HC0 e HC1 da inclinação e confira-os com o statsmodels. Compare os intervalos de 95% com o erro-padrão usual e com o HC1.

In [ ]:
# Escreva sua solução aqui

## Exercício 03 — Viés e incerteza no laboratório

Sorteie os índices de 5.000 amostras de 50 lotes com `rng.choice(N, size=(5_000, n), replace=True)` — uma linha por amostra — e, na versão crescente, calcule em cada amostra a inclinação e os três erros-padrão. Compare a média das inclinações com a inclinação da população e o desvio-padrão real das estimativas com a média dos erros-padrão usuais e HC1.

In [ ]:
# Escreva sua solução aqui

## Exercício 04 — Cobertura

Com as mesmas amostras, calcule a fração de intervalos de 95% que contêm a inclinação da população, com o erro-padrão usual e com o HC1. Use o valor crítico da $t$ com $n - 2$ graus de liberdade. Interprete.

In [ ]:
# Escreva sua solução aqui

## Exercício 05 — E quando a variância é constante?

Repita Exercício 04 com os mesmos índices na versão **constante**, a do Capítulo 6. O que acontece com a cobertura dos intervalos usuais e dos robustos? O que isso diz sobre usar erros-padrão robustos como padrão?

In [ ]:
# Escreva sua solução aqui

## Exercício 06 — Usual grande demais

Repita a comparação na versão **maior no meio**: média dos erros-padrão usuais e HC1, desvio-padrão real das estimativas e coberturas. O erro-padrão usual erra para mais ou para menos? Explique o resultado com os pesos $(x_i - \bar{x})^2$ da fórmula robusta.

In [ ]:
# Escreva sua solução aqui

# Exercícios 07–12 · Heterocedasticidade e erros-padrão robustos

**Prof. Daniel de Abreu Pereira Uhr**

**Curso:** Estatística e Econometria com Python


## Contexto — Imobiliária

Você volta aos 300 apartamentos da imobiliária do Capítulo 10: **aluguel** mensal (R\$), **área** (m²), **idade** do imóvel (anos) e **distância** ao centro (km). A célula abaixo reproduz os dados com o mesmo código e a mesma seed. Agora, a pergunta é sobre a precisão das estimativas.

Execute a célula abaixo antes de começar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

rng = np.random.default_rng(2026)
n_apartamentos = 300
apartamentos = pd.DataFrame({
    "area_m2": rng.integers(30, 151, size=n_apartamentos),
    "idade_anos": rng.integers(0, 41, size=n_apartamentos),
    "distancia_km": np.round(rng.uniform(0.5, 20, size=n_apartamentos), 1),
})
log_aluguel = (4.5 + 0.8 * np.log(apartamentos["area_m2"]) - 0.01 * apartamentos["idade_anos"]
               - 0.15 * np.log(apartamentos["distancia_km"]) + rng.normal(0, 0.15, size=n_apartamentos))
apartamentos["aluguel"] = np.round(np.exp(log_aluguel), -1)

## Exercício 07 — Diagnóstico

Estime a regressão do aluguel, em reais, na área, na idade e na distância. Faça o gráfico dos resíduos contra os valores ajustados e calcule o desvio-padrão dos resíduos nas faixas de área de 30 a 60, 60 a 90, 90 a 120 e 120 a 150 m². O que você conclui?

In [ ]:
# Escreva sua solução aqui

## Exercício 08 — Usuais e robustos lado a lado

Reestime o modelo com `cov_type="HC1"` e monte uma tabela com os erros-padrão usuais, os HC1 e a razão entre eles, para cada coeficiente. Qual erro-padrão mais mudou? Algum diminuiu?

In [ ]:
# Escreva sua solução aqui

## Exercício 09 — Intervalos e conclusões

Compare os intervalos de 95% da distância com os dois erros-padrão e interprete o coeficiente com unidade. Alguma conclusão sobre a área, a idade ou a distância muda com os erros-padrão robustos?

In [ ]:
# Escreva sua solução aqui

## Exercício 10 — O modelo em logaritmo

Estime o modelo em logaritmo do Capítulo 10 — `np.log(aluguel)` em `np.log(area_m2)`, `idade_anos` e `np.log(distancia_km)` — com erros-padrão usuais e HC1. Compare as razões entre os erros-padrão e o desvio-padrão dos resíduos nas mesmas faixas de área.

In [ ]:
# Escreva sua solução aqui

## Exercício 11 — Forma funcional ou remédio?

Um colega conclui: “o logaritmo resolveu a heterocedasticidade; então, sempre que houver heterocedasticidade, devemos usar o log”. Avalie essa afirmação.

In [ ]:
# Escreva sua solução aqui

## Exercício 12 — Descartar os grandes

Outro colega propõe “resolver” a heterocedasticidade do modelo em reais descartando os apartamentos com mais de 120 m². Reestime o modelo sem eles, com HC1, e avalie a proposta: quantos apartamentos saem, o que muda nas estimativas e nos erros-padrão e o que muda na pergunta que a análise responde.

In [ ]:
# Escreva sua solução aqui

# Exercícios 13–18 · Heterocedasticidade e erros-padrão robustos

**Prof. Daniel de Abreu Pereira Uhr**

**Curso:** Estatística e Econometria com Python


## Contexto — Distribuidora Falcão

A Falcão, distribuidora fictícia de alimentos, **sorteou** o número de visitas mensais de vendedores a 240 clientes — 1, 2, 4 ou 8 visitas, com probabilidades de 50%, 25%, 15% e 10% — e registrou as **compras** mensais de cada cliente (R\$ mil). Os dados são didáticos.

Cada visita custa R\$ 350, e a margem bruta da distribuidora é de 20%. Uma visita se paga se aumentar as compras do cliente em mais de R\$ 350 / 0,20 = **R\$ 1,75 mil** por mês.

O relatório da equipe comercial, feito com a tabela de regressão padrão, conclui que “as visitas se pagam com folga” e recomenda ampliá-las.

Execute a célula abaixo antes de começar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

rng = np.random.default_rng(2026)
n_clientes = 240
visitas = rng.choice([1, 2, 4, 8], size=n_clientes, p=[0.5, 0.25, 0.15, 0.10])
resposta_cliente = 2.0 + rng.normal(0, 1.5, size=n_clientes)
clientes = pd.DataFrame({
    "visitas": visitas,
    "compras": np.round(30 + resposta_cliente * visitas + rng.normal(0, 4, size=n_clientes), 1),
})

## Exercício 13 — A conclusão do relatório

Estime a regressão das compras no número de visitas com os erros-padrão usuais. Interprete o coeficiente com unidade e compare o intervalo de 95% com o ponto de equilíbrio de R\$ 1,75 mil. Reproduza a conclusão do relatório.

In [ ]:
# Escreva sua solução aqui

## Exercício 14 — Diagnóstico

Calcule o número de clientes e o desvio-padrão dos resíduos para cada número de visitas e faça o gráfico dos resíduos contra as visitas. O que você observa?

In [ ]:
# Escreva sua solução aqui

## Exercício 15 — Erros-padrão robustos

Reestime o modelo com `cov_type="HC1"`. Compare o erro-padrão e o intervalo de 95% com os de Exercício 13 e com o ponto de equilíbrio.

In [ ]:
# Escreva sua solução aqui

## Exercício 16 — Teste do ponto de equilíbrio

Teste $H_0: \beta_{visitas} = 1{,}75$ com os dois modelos, usando `t_test`. O que muda?

In [ ]:
# Escreva sua solução aqui

## Exercício 17 — Sorteio e heterocedasticidade

Por que o sorteio permite ler o coeficiente como o efeito médio de uma visita a mais? Por que, mesmo com sorteio, a dispersão das compras cresce com o número de visitas? Os erros-padrão robustos mudam a leitura causal?

In [ ]:
# Escreva sua solução aqui

## Exercício 18 — Recomendação

Escreva uma recomendação à diretoria sobre ampliar as visitas, com a estimativa, o intervalo robusto, a comparação com o ponto de equilíbrio e os limites. Proponha um próximo passo que reduza a incerteza.

In [ ]:
# Escreva sua solução aqui

### Exercício 18 · Decisão sobre a rota

**Sua decisão.** Nos exercícios 13–18, reproduza a análise e entregue um parecer para um segundo cenário. Suponha que uma rota custe R\$ 400 por visita, tenha margem de 15% e disponibilidade para 60 visitas adicionais. Informe o ponto de equilíbrio, a contribuição estimada para o lote de visitas e o intervalo associado à inclinação média, mantendo as hipóteses de custo e margem. Explique que informação adicional a Falcão deveria levantar antes de comprometer a rota.